# 第41章：无监督学习

本 Notebook 复算正文第8节：手算主成分与滤波、训练内 PCA 与全样本 PCA、解释方差与预测力、以及状态择时的成本后比较。核心逻辑来自 `quant_course.unsupervised_tools`。全部数据为**教学构造**，不是市场证据。

时间约定：$t$ 收盘形成信号（只用 $\le t$ 的数据，泄漏对照除外），$t+1$ 收盘成交，赚取 $t+2$ 的收益，单边 10 个基点。

读者包离线路径：教学构造或明确标注的接受历史摘要。新运行输出位于唯一 `runs/` 子目录；不构成真实市场研究、P6验证或A10-B执行。


In [ ]:
import os, sys
from pathlib import Path
from uuid import uuid4
candidate = Path.cwd().resolve()
ROOT = next((p for p in (candidate, *candidate.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start from the reader package root or a subdirectory")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("ch41_unsupervised_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
workspace_root = ROOT
print("Run directory:", RUN_DIR.relative_to(ROOT))
import importlib.util
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def load_chapter(name):
    spec = importlib.util.spec_from_file_location(name, ROOT / "code" / f"{name}.py")
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

from quant_course import unsupervised_tools as ut
ch41 = load_chapter('ch41_unsupervised')

## 1. 手算（第8.1、8.3节）

In [ ]:
hp, hh = ch41.hand_pca(), ch41.hand_hmm()
display(pd.DataFrame({k: {"eigenvalues": v["eigenvalues"], "PC1": [round(r[0], 4) for r in v["loadings"]],
                          "ratio": v["explained_ratio"]} for k, v in hp.items()}).T)
assert hp["cov_2_1_1_2"]["eigenvalues"] == [3.0, 1.0] and hp["cov_2_1_1_2"]["loadings"][0][1] > 0 > hp["cov_2_1_1_2"]["loadings"][1][1]
print(hh["posterior"])
assert np.allclose(hh["posterior"]["-0.025"], [0.057059, 0.942941], atol=1e-6)

## 2. 训练内 PCA vs 全样本 PCA（第8.2节）

In [ ]:
R, B = ut.simulate_factor_panel(1000, 50, factor_vols=(0.010, 0.005, 0.003), idio_vol=0.015, seed=4101)
noise = pd.DataFrame(np.random.default_rng(4103).normal(0, 0.015, (1000, 50)), index=R.index, columns=R.columns)
pca = {(p, w): ch41.rolling_pca(panel, w) for p, panel in (("factor", R), ("noise", noise)) for w in (60, 250)}
display(pd.DataFrame({f"{p} T={w}": {k: v[k] for k in ("in_sample_explained_mean", "oos_r2_mean", "full_sample_fit_on_test_r2_mean")}
                      for (p, w), v in pca.items()}).T.round(3))
assert round(pca[("noise", 60)]["in_sample_explained_mean"], 3) == 0.186 and round(pca[("noise", 250)]["oos_r2_mean"], 3) == 0.060
labels_ = [f"{p}\nT={w}" for p, w in pca]
xs = np.arange(len(pca))
plt.bar(xs - 0.2, [v["in_sample_explained_mean"] for v in pca.values()], 0.4, color="#7F7F7F", label="in-sample (k=3)")
plt.bar(xs + 0.2, [v["oos_r2_mean"] for v in pca.values()], 0.4, color="#247F82", label="out-of-sample R2")
plt.xticks(xs, labels_); plt.ylabel("share of variance"); plt.legend(); plt.title("CONSTRUCTED 50-asset panels"); plt.savefig(RUN_DIR / "ch41_unsupervised_cell05.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()

## 3. 解释方差不等于预测力（第6.4节、第12.5节）

In [ ]:
vp = ch41.variance_vs_prediction()
display(pd.DataFrame(vp).T)
assert vp["cov_k1"]["oos_r2"] < 0.01 < vp["corr_k1"]["oos_r2"] < vp["cov_k2"]["oos_r2"]

## 4. 状态择时（第8.4节）

名称带 `[leaks]`、`[impossible]`、`[oracle params]` 的规则使用了交易时不可得的信息，只作对照。

In [ ]:
reg = ch41.regime_case()
display(pd.DataFrame(reg["rules"]).T[["ann_return", "sharpe", "max_drawdown", "turnover_ann", "state_hit_rate"]].round(3))
assert round(reg["rules"]["filtered_prob_lagged"]["sharpe"], 2) == -0.51
assert reg["rules"]["smoothed_prob_same_day [leaks]"]["sharpe"] > reg["rules"]["smoothed_prob_lagged [leaks]"]["sharpe"]
idx = reg["_series"]["test_idx"]
for name in ("buy_and_hold", "smoothed_prob_same_day [leaks]", "filtered_prob_lagged", "trailing_vol_lagged"):
    plt.plot(idx, reg["_curves"][name], label=name)
plt.yscale("log"); plt.xlabel("date (test)"); plt.ylabel("wealth after 10 bp (log)"); plt.legend(fontsize=7)
plt.title("CONSTRUCTED regime market, seed 4102"); plt.savefig(RUN_DIR / "ch41_unsupervised_cell09.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()

## 5. 50 个种子的稳健性诊断（事后添加，D11）

In [ ]:
mc = ch41.regime_monte_carlo()
display(pd.DataFrame({k: v for k, v in mc.items() if k != "n_seeds"}).T.round(2))
assert round(mc["smoothed_prob_same_day [leaks]"]["median_sharpe"], 2) == 0.76 and round(mc["filtered_prob_lagged"]["median_sharpe"], 2) == -0.03